# Gene ↔ MolecularFunction Relation-Wise Merge

Merges Gene–MolecularFunction triples from Monarch, DRKG, Hetionet, GPKG, TARKG, and
Harmonizome; resolves missing gene head names via NCBI synonyms; deduplicates by
`(head, relation, tail)`; and saves the result.

## 0. Configuration

In [1]:

import os
import pandas as pd
import numpy as np

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR  = BASE_DIR + 'processed_data/'

OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/GENE_MOLECULARFUNCTION/ALL_GENE_MOLECULARFUNCTION.parquet'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Gene Lookup Dictionaries — NCBI and ENSEMBL

In [2]:
ENS_2NCBI = pd.read_csv(DB_DIR + 'ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv')
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))
del ENS_2NCBI

NCBI_ALL_GENE = pd.read_csv(DB_DIR + 'ncbi/Homo_sapiens.gene_info', sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict   = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['Symbol']))
NCBI_ALL_Symb_Desc_dict  = dict(zip(NCBI_ALL_GENE['Symbol'], NCBI_ALL_GENE['description']))

# Exploded synonym → canonical Symbol dict
NCBI_ALL_GENE_Syn_Dict = dict(zip(NCBI_ALL_GENE['Synonyms'], NCBI_ALL_GENE['Symbol']))
exploded_dict_NCBI_ALL_GENE_Syn_Dict = {}
for k, v in NCBI_ALL_GENE_Syn_Dict.items():
    for alias in k.split('|'):
        exploded_dict_NCBI_ALL_GENE_Syn_Dict[alias.strip()] = v

print(f"NCBI gene table: {len(NCBI_ALL_GENE):,} rows")
print(f"Synonym dict:    {len(exploded_dict_NCBI_ALL_GENE_Syn_Dict):,} entries")

NCBI gene table: 193,505 rows
Synonym dict:    69,564 entries


## 2. Load KG Sources

### 2a. Monarch

In [3]:
Monarch_Gene_MF = pd.read_csv(PROC_DIR + 'Monarch/Human/Gene_Human_MolecularActivity_MONARCH.csv')
Monarch_Gene_MF.columns = Monarch_Gene_MF.columns.str.lower()
Monarch_Gene_MF.rename(columns={'tail_name': 'tail_detail_name'}, inplace=True)
print(f"Monarch:     {len(Monarch_Gene_MF):,} rows")

Monarch_Gene_MF['kg_type'] = 'Generalised'
Monarch_Gene_MF['kg_source'] = 'Monarch_KG'
Monarch_Gene_MF['species'] = 'Homo species'

Monarch_Gene_MF.head(2)

Monarch:     347,707 rows


,head,relation,tail,head_type,relation_type,tail_type,relation_source,kg_source,head_detail_name,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,head_name,tail_detail_name,head_orignal,species_species,kg_type,species
0,SLC25A29,Gene_MolecularActivity,GO:0005292,Gene,enables,MolecularActivity,infores:go-central,Monarch_KG,solute carrier family 25 member 29,Homo sapiens,NaN,NCBI_ID,GO,SLC25A29,high-affinity lysine transmembrane transporter...,HGNC:20116,Homo sapiens_Homo sapiens,Generalised,Homo species
1,SMPDL3B,Gene_MolecularActivity,GO:0008081,Gene,enables,MolecularActivity,infores:go-central,Monarch_KG,sphingomyelin phosphodiesterase acid like 3B,Homo sapiens,NaN,NCBI_ID,GO,SMPDL3B,phosphoric diester hydrolase activity,HGNC:21416,Homo sapiens_Homo sapiens,Generalised,Homo species


### 2b. DRKG

In [4]:
DRKG_Gene_MF = pd.read_csv(PROC_DIR + 'DRKG/DRKG_Gene_Mol_Func.csv')
DRKG_Gene_MF.columns = DRKG_Gene_MF.columns.str.lower()
print(f"DRKG:        {len(DRKG_Gene_MF):,} rows")

DRKG_Gene_MF['kg_type'] = 'Generalised'
DRKG_Gene_MF['kg_source'] = 'DRKG'
DRKG_Gene_MF['species'] = 'Homo species'

DRKG_Gene_MF.head(2)

DRKG:        86,685 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_name,head_detail_name,head_ens,tail_detail_name,head_id_is,tail_id_is,head_orignal,tail_orignal,kg_type,species
0,PAXX,Gene_Molecular Function,GO:0042803,Gene,Hetionet::GpMF::Gene:Molecular Function,Molecular Function,DRKG,286257,PAXX non-homologous end joining factor,ENSG00000148362,protein homodimerization activity,NCBI_ID,Quick_GO,Gene::286257,Molecular Function::GO:0042803,Generalised,Homo species
1,PRMT5,Gene_Molecular Function,GO:0016274,Gene,Hetionet::GpMF::Gene:Molecular Function,Molecular Function,DRKG,10419,protein arginine methyltransferase 5,ENSG00000100462,protein-arginine N-methyltransferase activity,NCBI_ID,Quick_GO,Gene::10419,Molecular Function::GO:0016274,Generalised,Homo species


### 2c. Hetionet

In [5]:
Hetionet_Gene_MF = pd.read_csv(PROC_DIR + 'Hetionet/Gene_MolecularFunction_Hetionet.csv')
Hetionet_Gene_MF.columns = Hetionet_Gene_MF.columns.str.lower()
print(f"Hetionet:    {len(Hetionet_Gene_MF):,} rows")
Hetionet_Gene_MF['kg_type'] = 'Generalised'
Hetionet_Gene_MF['kg_source'] = 'Hetionet'
Hetionet_Gene_MF['species'] = 'Homo species'
Hetionet_Gene_MF.head(2)

Hetionet:    97,208 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_name,head_detail_name,tail_detail_name,kg_type,species
0,PAXX,Gene_MolecularFunction,GO:0042803,Gene,Gene–participates–MolecularFunction,Molecular Function,Hetionet,NCBI_ID,Quick_GO,286257,PAXX non-homologous end joining factor,protein homodimerization activity,Generalised,Homo species
1,PRMT5,Gene_MolecularFunction,GO:0016274,Gene,Gene–participates–MolecularFunction,Molecular Function,Hetionet,NCBI_ID,Quick_GO,10419,protein arginine methyltransferase 5,protein-arginine N-methyltransferase activity,Generalised,Homo species


### 2d. GPKG

In [6]:
GPKG_Gene_MF = pd.read_csv(PROC_DIR + 'GPKG/Gene_MolecularFunction_GPKG.csv')
GPKG_Gene_MF.columns = GPKG_Gene_MF.columns.str.lower()
print(f"GPKG:        {len(GPKG_Gene_MF):,} rows")
GPKG_Gene_MF['kg_type'] = 'Generalised'
GPKG_Gene_MF['kg_source'] = 'GPKG'
GPKG_Gene_MF['species'] = 'Homo species'
GPKG_Gene_MF.head(2)

GPKG:        43,147 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_orignal,head_alias_mapped,head_detail_name,head_id_is,tail_detail_name,tail_id_is,kg_type,species
0,CYP2D7,Gene_MolecularFunction,GO:0016712,Gene,associated,MolecularFunction,GPKG,ENSG00000205702,CYP2D7,cytochrome P450 family 2 subfamily D member 7 ...,NCBI_ID,"oxidoreductase activity, acting on paired dono...",Quick_GO,Generalised,Homo species
1,CYP2D7,Gene_MolecularFunction,GO:0008395,Gene,associated,MolecularFunction,GPKG,ENSG00000205702,CYP2D7,cytochrome P450 family 2 subfamily D member 7 ...,NCBI_ID,steroid hydroxylase activity,Quick_GO,Generalised,Homo species


### 2e. TARKG

In [7]:
TARKG_Gene_MF = pd.read_csv(PROC_DIR + 'TARKG/Gene_MolecularFunction_TARKG.csv')
TARKG_Gene_MF.columns = TARKG_Gene_MF.columns.str.lower()
print(f"TARKG:       {len(TARKG_Gene_MF):,} rows")

TARKG_Gene_MF['kg_type'] = 'Generalised'
TARKG_Gene_MF['kg_source'] = 'TARKG'
TARKG_Gene_MF['species'] = 'Homo species'

TARKG_Gene_MF.head(2)

TARKG:       166,778 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,CREBBP,Gene_Molecular Function,GO:0000981,Gene,GpMF,Molecular Function,TARKG,CREB binding protein,"DNA-binding transcription factor activity, RNA...",NCBI_ID,Quick_GO,Hetionet-1075588,Hetionet,0,Generalised,Homo species
1,CREBBP,Gene_Molecular Function,GO:0000981,Gene,Hetionet::GpMF::Gene:Molecular Function,Molecular Function,TARKG,CREB binding protein,"DNA-binding transcription factor activity, RNA...",NCBI_ID,Quick_GO,DRKG-2946790,DRKG,0,Generalised,Homo species


### 2f. Harmonizome

In [8]:
Harmonizome_Gene_MF = pd.read_csv(PROC_DIR + 'harmonizome/Gene_MolecularFunction_Harmonizome.csv')
Harmonizome_Gene_MF.columns = Harmonizome_Gene_MF.columns.str.lower()
print(f"Harmonizome: {len(Harmonizome_Gene_MF):,} rows")
Harmonizome_Gene_MF['kg_type'] = 'Generalised'
Harmonizome_Gene_MF['kg_source'] = 'Harmonizome'
Harmonizome_Gene_MF['species'] = 'Homo species'
Harmonizome_Gene_MF.head(2)

Harmonizome: 1,078,295 rows


,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,ABCA1,Gene_MolecularFunction,GO:0042626,Gene,MolecularFunction,GO MF,Harmonizome,ATP binding cassette subfamily A member 1,ATPase-coupled transmembrane transporter activity,NCBI_ID,Quick_GO,Generalised,Homo species
1,ABCA10,Gene_MolecularFunction,GO:0042626,Gene,MolecularFunction,GO MF,Harmonizome,ATP binding cassette subfamily A member 10,ATPase-coupled transmembrane transporter activity,NCBI_ID,Quick_GO,Generalised,Homo species


## 3. Check and Fix Duplicate Columns

In [9]:
SOURCE_DFS = [
    ('Monarch_Gene_MF',     Monarch_Gene_MF),
    ('DRKG_Gene_MF',        DRKG_Gene_MF),
    ('Hetionet_Gene_MF',    Hetionet_Gene_MF),
    ('GPKG_Gene_MF',        GPKG_Gene_MF),
    ('TARKG_Gene_MF',       TARKG_Gene_MF),
    ('Harmonizome_Gene_MF', Harmonizome_Gene_MF),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[Monarch_Gene_MF] ✓ no duplicates
[DRKG_Gene_MF] ✓ no duplicates
[Hetionet_Gene_MF] ✓ no duplicates
[GPKG_Gene_MF] ✓ no duplicates
[TARKG_Gene_MF] ✓ no duplicates
[Harmonizome_Gene_MF] ✓ no duplicates


## 4. Align Schemas and Concatenate

In [10]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 1,819,820 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,SLC25A29,Gene_MolecularActivity,GO:0005292,Gene,enables,MolecularActivity,Monarch_KG,NCBI_ID,GO,solute carrier family 25 member 29,high-affinity lysine transmembrane transporter...,Generalised,Homo species
1,SMPDL3B,Gene_MolecularActivity,GO:0008081,Gene,enables,MolecularActivity,Monarch_KG,NCBI_ID,GO,sphingomyelin phosphodiesterase acid like 3B,phosphoric diester hydrolase activity,Generalised,Homo species


## 5. Standardise Fixed-Value Columns

In [11]:
final_df['relation']   = 'Gene_MolecularFunction'
final_df['head_type']  = 'Gene'
final_df['tail_type']  = 'MolecularFunction'
final_df['head_id_is'] = 'NCBI_ID'
final_df['tail_id_is'] = 'Quick_GO'

print("Unique relation:",  set(final_df['relation']))
print("Unique head_type:", set(final_df['head_type']))
print("Unique tail_type:", set(final_df['tail_type']))
print("Unique kg_source:", set(final_df['kg_source']))

Unique relation: {'Gene_MolecularFunction'}
Unique head_type: {'Gene'}
Unique tail_type: {'MolecularFunction'}
Unique kg_source: {'TARKG', 'Harmonizome', 'Monarch_KG', 'DRKG', 'GPKG', 'Hetionet'}


## 6. Deduplicate

In [12]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 1,130,170 rows


## 7. Repair Missing Gene Head Names

For rows missing `head_detail_name`: strip `-`, map synonyms → canonical symbol,
then NCBI Symbol → description.

In [13]:
mask = final_df_dedup['head_detail_name'].isna()
print(f"Rows needing head_detail_name repair: {mask.sum():,}")

# Clean symbol (remove '-') then map synonyms → canonical symbol
final_df_dedup.loc[mask, 'head'] = final_df_dedup.loc[mask, 'head'].str.replace('-', '', regex=False)
final_df_dedup.loc[mask, 'head'] = (
    final_df_dedup.loc[mask, 'head']
    .map(exploded_dict_NCBI_ALL_GENE_Syn_Dict)
    .fillna(final_df_dedup.loc[mask, 'head'])
)

# NCBI Symbol → description
mask2 = final_df_dedup['head_detail_name'].isna()
final_df_dedup.loc[mask2, 'head_detail_name'] = final_df_dedup.loc[mask2, 'head'].map(NCBI_ALL_Symb_Desc_dict)
print(f"Still missing head_detail_name: {final_df_dedup['head_detail_name'].isna().sum():,}")

Rows needing head_detail_name repair: 39
Still missing head_detail_name: 0


## 8. Add Schema Columns and Enforce Column Order

In [14]:

final_df_dedup = final_df_dedup[REQUIRED_COLS]
print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

Final shape: (1130170, 13)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,A1BG,Gene_MolecularFunction,GO:0003674,Gene,None,MolecularFunction,Harmonizome,NCBI_ID,Quick_GO,alpha-1-B glycoprotein,molecular_function,Generalised,Homo species
1,A1BG,Gene_MolecularFunction,GO:0008150,Gene,None,MolecularFunction,Harmonizome,NCBI_ID,Quick_GO,alpha-1-B glycoprotein,biological_process,Generalised,Homo species
2,A1CF,Gene_MolecularFunction,GO:0000166,Gene,None,MolecularFunction,Harmonizome,NCBI_ID,Quick_GO,APOBEC1 complementation factor,nucleotide binding,Generalised,Homo species
3,A1CF,Gene_MolecularFunction,GO:0003674,Gene,None,MolecularFunction,Harmonizome,NCBI_ID,Quick_GO,APOBEC1 complementation factor,molecular_function,Generalised,Homo species
4,A1CF,Gene_MolecularFunction,GO:0003676,Gene,None,MolecularFunction,Harmonizome,NCBI_ID,Quick_GO,APOBEC1 complementation factor,nucleic acid binding,Generalised,Homo species


## 9. QC — NaN Check

In [15]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,985305,0,985305
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [16]:
final_df_dedup[final_df_dedup['species'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species


## 10. Save Output

In [17]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_parquet(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 1,130,170 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/GENE_MOLECULARFUNCTION/ALL_GENE_MOLECULARFUNCTION.parquet


In [19]:
#